# Experiment 4: Term Frequency Analysis, Named Entity Recognition, and TF-IDF

This notebook demonstrates vocabulary frequency analysis using an NLP toolkit (spaCy) and pure Python, extracts Named Entities (NER), and implements a complete multi-document Term Frequency - Inverse Document Frequency (TF-IDF) mathematical pipeline from scratch.

## 1. Setup and Environment Imports

Import required standard libraries (`string`, `math`, `collections.Counter`), data processing tool (`pandas`), and NLP pipeline (`spacy`).

In [1]:
import string
import math
import pandas as pd
from collections import Counter
import spacy

nlp = spacy.load('en_core_web_sm')

## Experiment 4.1: Term-Frequency Analysis and Named Entity Recognition Using an NLP Toolkit

Perform word frequency counting using spaCy tokenization, save term frequencies to CSV, display the top 10 terms, and extract named entities.

In [2]:
with open('4.1_4.2_input.txt', 'r', encoding='utf-8') as f:
    text_4_1 = f.read()

doc_4_1 = nlp(text_4_1)
tokens_toolkit = [token.text.lower() for token in doc_4_1 if token.is_alpha]
tf_toolkit = Counter(tokens_toolkit)

df_tf_toolkit = pd.DataFrame(tf_toolkit.most_common(), columns=['Term', 'Frequency'])
df_tf_toolkit.to_csv('4.1_term_frequency_toolkit.csv', index=False)

print("Top 10 Most Frequent Terms (Toolkit):")
print(df_tf_toolkit.head(10))

entities_list = [(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc_4_1.ents]
df_ner = pd.DataFrame(entities_list, columns=['Entity', 'Label', 'Description']).drop_duplicates()

print("\nNamed Entity Recognition (NER) Results Sample:")
print(df_ner.head(10))

Top 10 Most Frequent Terms (Toolkit):
   Term  Frequency
0     a         88
1   and         86
2   the         75
3    of         42
4    to         42
5    in         42
6   can         42
7    is         37
8   may         30
9  word         30

Named Entity Recognition (NER) Results Sample:
                                               Entity     Label  \
0   Natural Language Processing and Artificial Int...       ORG   
1                                                 NLP       ORG   
2                                             English  LANGUAGE   
3                                               Hindi       GPE   
4                                            Assamese      NORP   
5                                             Bengali      NORP   
6                                               Tamil       GPE   
10                                             twenty  CARDINAL   
11                                             twenty      DATE   
12                                 

## Experiment 4.2: Term-Frequency Analysis Without Using Any NLP Toolkit (Pure Python)

Calculate word frequencies using standard Python string manipulation and dictionaries, export results to CSV, and identify the top 10 most frequent terms.

In [3]:
with open('4.1_4.2_input.txt', 'r', encoding='utf-8') as f:
    text_4_2 = f.read()

clean_text_4_2 = text_4_2.lower().translate(str.maketrans('', '', string.punctuation))
words_pure = [w for w in clean_text_4_2.split() if w.isalpha()]

tf_dict_pure = {}
for word in words_pure:
    tf_dict_pure[word] = tf_dict_pure.get(word, 0) + 1

sorted_tf_pure = sorted(tf_dict_pure.items(), key=lambda item: item[1], reverse=True)
df_tf_pure = pd.DataFrame(sorted_tf_pure, columns=['Term', 'Frequency'])
df_tf_pure.to_csv('4.2_term_frequency_pure_python.csv', index=False)

print("Top 10 Most Frequent Terms (Pure Python):")
print(df_tf_pure.head(10))

Top 10 Most Frequent Terms (Pure Python):
   Term  Frequency
0     a         90
1   and         86
2   the         75
3    to         42
4    in         42
5   can         41
6    of         40
7    is         37
8   may         30
9  word         30


## Experiment 4.3: Manual Multi-Document TF-IDF Calculation From Scratch

Implement Term Frequency (TF), Document Frequency (DF), Inverse Document Frequency (IDF), and TF-IDF scoring across three documents without using any external NLP library.

In [4]:
documents = [
    "Natural language processing is a field of artificial intelligence.",
    "Natural language processing helps computers understand human language.",
    "Machine learning is an important part of artificial intelligence."
]

cleaned_docs = []
for doc in documents:
    no_punct = doc.lower().translate(str.maketrans('', '', string.punctuation))
    cleaned_docs.append(no_punct.split())

vocabulary = sorted(list(set(term for doc in cleaned_docs for term in doc)))
num_docs = len(documents)

df_counts = {}
for term in vocabulary:
    df_counts[term] = sum(1 for doc in cleaned_docs if term in doc)

idf_values = {}
for term in vocabulary:
    idf_values[term] = math.log(num_docs / df_counts[term])

df_stats = pd.DataFrame({
    'Term': vocabulary,
    'DF (Doc Frequency)': [df_counts[t] for t in vocabulary],
    'IDF (Inverse Doc Frequency)': [round(idf_values[t], 4) for t in vocabulary]
})

print("1. Document Frequency (DF) and Inverse Document Frequency (IDF):")
print(df_stats)

tfidf_records = []
for doc_idx, doc_tokens in enumerate(cleaned_docs, 1):
    doc_len = len(doc_tokens)
    term_counts = Counter(doc_tokens)
    for term in vocabulary:
        count = term_counts[term]
        tf = count / doc_len
        idf = idf_values[term]
        tfidf = tf * idf
        if count > 0:
            tfidf_records.append({
                'Document': f'Doc {doc_idx}',
                'Term': term,
                'Count': count,
                'TF': round(tf, 4),
                'IDF': round(idf, 4),
                'TF-IDF Score': round(tfidf, 4)
            })

df_tfidf_results = pd.DataFrame(tfidf_records)
print("\n2. Term Frequency (TF) and TF-IDF Scores for Present Terms:")
print(df_tfidf_results)

print("\n3. Top Terms with Highest TF-IDF Score Per Document:")
for doc_idx in range(1, num_docs + 1):
    doc_name = f'Doc {doc_idx}'
    subset = df_tfidf_results[df_tfidf_results['Document'] == doc_name]
    top_terms = subset.sort_values(by='TF-IDF Score', ascending=False)
    print(f"\n--- {doc_name} Top Terms ---")
    print(top_terms[['Term', 'TF-IDF Score']].head(5))

1. Document Frequency (DF) and Inverse Document Frequency (IDF):
            Term  DF (Doc Frequency)  IDF (Inverse Doc Frequency)
0              a                   1                       1.0986
1             an                   1                       1.0986
2     artificial                   2                       0.4055
3      computers                   1                       1.0986
4          field                   1                       1.0986
5          helps                   1                       1.0986
6          human                   1                       1.0986
7      important                   1                       1.0986
8   intelligence                   2                       0.4055
9             is                   2                       0.4055
10      language                   2                       0.4055
11      learning                   1                       1.0986
12       machine                   1                       1.0986
13       na